# Story day 1-20 analysis

In [1]:
# hide-output
# Import libraries and initialise the BigQuery connector

# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np
import plotly.graph_objects as go

bqc = BigQueryConnector()

<!-- hide --> 
## Aux functions

In [2]:
# hide-output
# Helper functions for weighted progression, percentile calculation, level visualisations, and retention significance
from aux_functions import (
    compute_weighted_progression,
    weighted_quantiles,
    add_event_annotations,
    add_median_lines,
    plot_percentile_comparison,
    compute_retention_significance,
    plot_retention_significance,
)

## Get data

### Player level and game day

In [3]:
# Compute symmetric A/B date windows of equal length anchored on 2026-06-01 (FTUE launch date)

import datetime as dt

new_ftue_date = dt.datetime(2026, 7, 1)
days_from_start = (dt.datetime.today() - new_ftue_date).days
#days_from_start = 28  # Set the number of days for the first window
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
#end_date2 = dt.datetime.today()-dt.timedelta(days=1)
end_date2 = new_ftue_date + dt.timedelta(days=days_from_start-1)

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

Start Date 1: 2026-05-19
End Date 1: 2026-06-30
Start Date 2: 2026-07-01
End Date 2: 2026-08-12


In [4]:
# hide-output
# Toggle: True re-queries BigQuery and overwrites local cache, False loads from pickle
refresh_data = True

In [5]:
# hide-output
# Estimate query cost for player level + game day SQL before executing

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE','RTG']
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 31.92 GB when run.
Estimated query cost: $0.21


In [6]:
# hide-output
# Fetch player level + game day data from BigQuery or load from local pickle cache
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [7]:
# hide-output
# Preview raw player level and game day data
data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version
0,4D0D930275F288D5,2026-08-07,2026-08-05,CA,2026-08-02,2026-08-01,2,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0
1,891692CD0FACEB18,2026-08-09,2026-08-09,US,2026-08-09,2026-08-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.81.0
2,2D0AB22D5B135786,2026-08-06,2026-08-05,DE,2026-08-02,2026-08-01,1,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0
3,F75214BDE9929B0A,2026-08-08,2026-08-02,FR,2026-08-02,2026-08-01,6,6,4,IOS,Non-Attributed,Non-Attributed,0.80.0
4,285D0B14A1C4AEB,2026-08-05,2026-08-05,BR,2026-08-02,2026-08-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.80.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
296549,ED4C793E1996E209,2026-08-04,2026-07-29,DE,2026-07-26,2026-07-01,6,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0
296550,5D45FDE6D033ED4E,2026-07-30,2026-07-29,US,2026-07-26,2026-07-01,1,9,6,IOS,Non-Attributed,Non-Attributed,0.80.0
296551,6049F381B9B68CBE,2026-08-02,2026-07-29,US,2026-07-26,2026-07-01,4,8,5,IOS,Non-Attributed,Non-Attributed,0.80.0
296552,5F4E2C7AE2C61852,2026-08-08,2026-07-29,DE,2026-07-26,2026-07-01,10,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0


<!-- hide --> 
### Retention

In [8]:
# hide-output
# Estimate query cost for per-install-date retention SQL
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE','RTG']          
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.25 GB when run.
Estimated query cost: $0.01


In [9]:
# hide-output
# Fetch per-install-date retention data from BigQuery or load from local pickle cache
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [10]:
# hide-output
# Sort retention data and spot-check Android rows
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

,install_dt,dx,platform,cohort_size,retained_size,retention_rate
0,2026-05-19,0,AND,279,279,1.000000
3,2026-05-19,1,AND,279,74,0.265233
5,2026-05-19,3,AND,279,50,0.179211
6,2026-05-19,7,AND,279,34,0.121864
8,2026-05-19,14,AND,279,23,0.082437
...,...,...,...,...,...,...
1095,2026-08-10,0,AND,255,255,1.000000
1097,2026-08-10,1,AND,255,56,0.219608
1099,2026-08-11,0,AND,234,234,1.000000
1100,2026-08-11,1,AND,234,49,0.209402


In [11]:
# hide-output
# Estimate query cost for FTUE-split retention SQL (all users)
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE','RTG']        
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.25 GB when run.
Estimated query cost: $0.01


In [12]:
# hide-output
# Fetch FTUE-split retention (all users) from BigQuery or load from local pickle cache
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [13]:
# hide-output
# Sort FTUE retention data and spot-check at D14
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
#retention_data_total[retention_data_total['dx'] == 14]
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,43,13096,13096,1.000000
0,0,AND,B.Post-FTUE revamp,43,10803,10803,1.000000
2,0,IOS,A.Pre-FTUE revamp,43,24231,24231,1.000000
3,0,IOS,B.Post-FTUE revamp,43,21668,21668,1.000000
4,1,AND,A.Pre-FTUE revamp,42,10651,3125,0.293400
5,1,AND,B.Post-FTUE revamp,42,8671,2543,0.293276
6,1,IOS,A.Pre-FTUE revamp,42,23491,7635,0.325018
7,1,IOS,B.Post-FTUE revamp,42,21126,6992,0.330967
8,3,AND,A.Pre-FTUE revamp,40,10076,1776,0.176260
9,3,AND,B.Post-FTUE revamp,40,8037,1529,0.190245


In [14]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
    #'exclude_networks': ['CPE','Non-Attributed']
    'exclude_networks': ['CPE','RTG']           
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.25 GB when run.
Estimated query cost: $0.01


In [15]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [16]:
# hide-output
# Sort and preview organic-only FTUE retention data
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,43,12284,12284,1.000000
0,0,AND,B.Post-FTUE revamp,43,10095,10095,1.000000
3,0,IOS,A.Pre-FTUE revamp,43,20258,20258,1.000000
2,0,IOS,B.Post-FTUE revamp,43,19162,19162,1.000000
4,1,AND,A.Pre-FTUE revamp,42,9904,2988,0.301696
5,1,AND,B.Post-FTUE revamp,42,8020,2416,0.301247
6,1,IOS,A.Pre-FTUE revamp,42,19597,6427,0.327958
7,1,IOS,B.Post-FTUE revamp,42,18638,6172,0.331151
9,3,AND,A.Pre-FTUE revamp,40,9371,1686,0.179917
8,3,AND,B.Post-FTUE revamp,40,7422,1448,0.195096


### AB metrics

In [17]:
# hide-output
# Estimate query cost for FTUE-split organic-only retention SQL
query_location = './sql/abmetrics.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),  
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 5.67 GB when run.
Estimated query cost: $0.04


In [18]:
# hide-output
# Fetch FTUE-split organic-only retention from BigQuery or load from local pickle cache
abmetrics = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    abmetrics = bqc.get(query='./sql/abmetrics.sql', is_path=True, query_parameters=parameters)
    abmetrics.to_pickle('./data/abmetrics.pkl')
else:
    # Load from local cache to avoid repeated query costs
    abmetrics = pd.read_pickle('./data/abmetrics.pkl')

In [19]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,4E457DC98BCE17B0,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
1,71519947AC3D8D6F,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
2,EF4D5A89B33B8146,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,4,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
3,B83F360FF8B1D488,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
4,8EE0EDAA76098BD1,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447440,93A74E043E9178B8,2026-05-30,2026-05-30,0,9,6,1,1,<NA>,7,...,0,0,0,0,0,6,0,1,0,2026-06-18 18:29:41.321007+00:00
447441,2BF08A70CB095BDA,2026-05-22,2026-05-30,8,18,13,1,9,1,9,...,0,0,0,0,0,2,0,1,0,2026-06-18 18:29:41.321007+00:00
447442,E347D9EFB0AC9BEC,2026-05-28,2026-05-30,2,14,10,1,3,1,26,...,0,0,0,0,0,2,0,1,0,2026-06-18 18:29:41.321007+00:00
447443,66A8C4D0719CE7E4,2026-05-29,2026-05-30,1,21,17,1,2,1,19,...,0,0,0,0,0,0,0,1,1,2026-06-18 18:29:41.321007+00:00


<!-- hide --> 
## Process data

In [20]:
# hide-output

days_from_install_limit = 28
# Assign FTUE flag, cap days_since_install to match B.new window, and drop immature cohort rows
dt_mode = 'install_dt'
processed_data = data.copy()

processed_data['install_dt'] = processed_data[dt_mode]

processed_data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in processed_data['acquisition_type']]
processed_data['FTUE_flag'] = ['B.new' if x >= new_ftue_date else 'A.old' for x in pd.to_datetime(processed_data['install_dt'])]

# Making comparison fair by capping the days_since_install for B.new to match the number of days since the new FTUE date.
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime(new_ftue_date)).days
processed_data = processed_data[~(processed_data['days_since_install'] > max_dayx_B_new)]

# Cap data to only days from install x or less for both cohorts to ensure fair comparison, building up on the above
processed_data = processed_data[processed_data['days_since_install'] <= days_from_install_limit]

processed_data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
processed_data = processed_data[processed_data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(processed_data['install_dt'])).dt.days - min_days_since_install]


# Add a country filter just to see if metrics align better with the US-only data. This is a temporary filter for testing purposes.
#processed_data = processed_data[processed_data['country_code'] == 'US']

processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,4D0D930275F288D5,2026-08-07,2026-08-05,CA,2026-08-02,2026-08-01,2,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
1,891692CD0FACEB18,2026-08-09,2026-08-09,US,2026-08-09,2026-08-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.81.0,N,B.new,dummy
2,2D0AB22D5B135786,2026-08-06,2026-08-05,DE,2026-08-02,2026-08-01,1,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
3,F75214BDE9929B0A,2026-08-08,2026-08-02,FR,2026-08-02,2026-08-01,6,6,4,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
4,285D0B14A1C4AEB,2026-08-05,2026-08-05,BR,2026-08-02,2026-08-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296549,ED4C793E1996E209,2026-08-04,2026-07-29,DE,2026-07-26,2026-07-01,6,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296550,5D45FDE6D033ED4E,2026-07-30,2026-07-29,US,2026-07-26,2026-07-01,1,9,6,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296551,6049F381B9B68CBE,2026-08-02,2026-07-29,US,2026-07-26,2026-07-01,4,8,5,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296552,5F4E2C7AE2C61852,2026-08-08,2026-07-29,DE,2026-07-26,2026-07-01,10,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy


In [21]:
# hide-output
# Sanity-check unique user counts per FTUE group
test = processed_data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,FTUE_flag,users
0,A.old,23402
1,B.new,19102


## Retention

In [22]:
# hide-output
# Add combined dx_platform column for the per-install-date retention line chart
retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']
retention_data

,install_dt,dx,platform,cohort_size,retained_size,retention_rate,combined_dimension
0,2026-05-19,0,AND,279,279,1.000000,0_AND
1,2026-05-19,0,IOS,570,570,1.000000,0_IOS
3,2026-05-19,1,AND,279,74,0.265233,1_AND
2,2026-05-19,1,IOS,570,180,0.315789,1_IOS
5,2026-05-19,3,AND,279,50,0.179211,3_AND
...,...,...,...,...,...,...,...
1098,2026-08-11,0,IOS,318,318,1.000000,0_IOS
1100,2026-08-11,1,AND,234,49,0.209402,1_AND
1101,2026-08-11,1,IOS,318,104,0.327044,1_IOS
1103,2026-08-12,0,AND,223,223,1.000000,0_AND


In [23]:
# Per-install-date retention rate over time by dx/platform (figure disabled — used for investigation only)
fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              facet_row='platform',
              width=1200,
              height=800,
              hover_data={'install_dt': True, 'retained_size': True},)

#fig.show()

In [24]:
# hide-output
# Preview FTUE-split retention totals (all users)
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,43,13096,13096,1.000000
0,0,AND,B.Post-FTUE revamp,43,10803,10803,1.000000
2,0,IOS,A.Pre-FTUE revamp,43,24231,24231,1.000000
3,0,IOS,B.Post-FTUE revamp,43,21668,21668,1.000000
4,1,AND,A.Pre-FTUE revamp,42,10651,3125,0.293400
5,1,AND,B.Post-FTUE revamp,42,8671,2543,0.293276
6,1,IOS,A.Pre-FTUE revamp,42,23491,7635,0.325018
7,1,IOS,B.Post-FTUE revamp,42,21126,6992,0.330967
8,3,AND,A.Pre-FTUE revamp,40,10076,1776,0.176260
9,3,AND,B.Post-FTUE revamp,40,8037,1529,0.190245


### Cohort sizes

In [25]:
# Bar chart: cohort sizes at each retention checkpoint by FTUE group and platform
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='cohort_size',
    color='FTUE_flag',
    text='cohort_size',
    title='Cohort sizes',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True, 'num_cohorts': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:0}', textposition='outside')
fig.update_layout(
    #yaxis_tickformat='.0%',
    #yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention rate 

In [26]:
# Bar chart: D1–D21 retention rates by FTUE group and platform (all users)
# Each Dx is tested independently with a two-proportion z-test; Wilson 95% CIs shown as error bars
dummy = plot_retention_significance(
    retention_data_total,
    title='Retention rate by FTUE group',
)

### Retention rate (Organics only)

In [27]:
# hide-output
# Bar chart: D1–D21 retention rates by FTUE group and platform (organic / non-attributed only)
# Each Dx is tested independently; small D21 organic cohorts annotated with users needed for significance
dummy = plot_retention_significance(
    retention_data_total_na,
    title='Retention rate by FTUE group — organic only',
)

## Player max level distribution

In [28]:
# hide-output
# Preview player data
processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,4D0D930275F288D5,2026-08-07,2026-08-05,CA,2026-08-02,2026-08-01,2,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
1,891692CD0FACEB18,2026-08-09,2026-08-09,US,2026-08-09,2026-08-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.81.0,N,B.new,dummy
2,2D0AB22D5B135786,2026-08-06,2026-08-05,DE,2026-08-02,2026-08-01,1,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
3,F75214BDE9929B0A,2026-08-08,2026-08-02,FR,2026-08-02,2026-08-01,6,6,4,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
4,285D0B14A1C4AEB,2026-08-05,2026-08-05,BR,2026-08-02,2026-08-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296549,ED4C793E1996E209,2026-08-04,2026-07-29,DE,2026-07-26,2026-07-01,6,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296550,5D45FDE6D033ED4E,2026-07-30,2026-07-29,US,2026-07-26,2026-07-01,1,9,6,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296551,6049F381B9B68CBE,2026-08-02,2026-07-29,US,2026-07-26,2026-07-01,4,8,5,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296552,5F4E2C7AE2C61852,2026-08-08,2026-07-29,DE,2026-07-26,2026-07-01,10,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy


In [29]:
# hide-output
# Build level funnel with P10/P50/P90 percentiles per FTUE group, platform, and day since install
days_since_install_limit = 21

pl_ftue_funnel_agg = processed_data.groupby(['max_gameday','FTUE_flag','platform', 'days_since_install']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag','platform','days_since_install']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on=['FTUE_flag','platform','days_since_install'])
pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby(['max_gameday','platform','days_since_install'])['pctg_users'].pct_change().fillna(0)

pl_ftue_funnel_agg['combined_dimension'] = pl_ftue_funnel_agg['FTUE_flag'].astype(str) + ' | ' + pl_ftue_funnel_agg['days_since_install'].astype(str)

# Calculate percentiles by FTUE_flag, platform, and days_since_install
level_dist = processed_data.groupby(['days_since_install', 'max_gameday', 'FTUE_flag', 'platform']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts_by_group = level_dist[level_dist.days_since_install<=days_since_install_limit].groupby(['days_since_install', 'FTUE_flag', 'platform']).apply(
    weighted_quantiles, measure_col='max_gameday', include_groups=False
).reset_index()

# Rename columns for clarity
#level_pcts_by_group = level_pcts_by_group.rename(columns={'p10': 'p10_max_gameday', 'p50': 'p50_max_gameday', 'p90': 'p90_max_gameday'})


# Merge into pl_ftue_funnel_agg
pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(
    level_pcts_by_group, 
    on=['days_since_install', 'FTUE_flag', 'platform'], 
    how='left'
)

pl_ftue_funnel_agg = pl_ftue_funnel_agg.loc[pl_ftue_funnel_agg['days_since_install'].isin([0,1,3,7,14,21,28])]


pl_ftue_funnel_agg

,max_gameday,FTUE_flag,platform,days_since_install,users,total_users,pctg_users,pctg_diff_users,combined_dimension,p10,p50,p90
0,1,A.old,AND,0,3052,6637,0.459846,0.0,A.old | 0,1.0,2.0,3.0
1,1,A.old,AND,1,343,2832,0.121116,0.0,A.old | 1,1.0,3.0,4.0
3,1,A.old,AND,3,91,1776,0.051239,0.0,A.old | 3,2.0,4.0,8.0
7,1,A.old,AND,7,38,1351,0.028127,0.0,A.old | 7,3.0,7.0,11.0
14,1,A.old,AND,14,20,1052,0.019011,0.0,A.old | 14,3.0,9.0,16.0
...,...,...,...,...,...,...,...,...,...,...,...,...
5264,206,B.new,IOS,3,4,3897,0.001026,0.0,B.new | 3,2.0,4.0,8.0
5268,206,B.new,IOS,7,5,2549,0.001962,0.0,B.new | 7,3.0,7.0,10.0
5275,206,B.new,IOS,14,3,1646,0.001823,0.0,B.new | 14,4.0,9.0,16.0
5282,206,B.new,IOS,21,1,1040,0.000962,0.0,B.new | 21,5.0,12.0,21.0


In [30]:
# hide-output
# Define level milestone annotations for A.old and B.new FTUE feature unlock points
events_config = {
    #'A.old': [
    #    {'level': 7, 'name': 'SP', 'color':'blue'},
    #    {'level': 8, 'name': 'Deco', 'color': 'blue'},
    #    {'level': 10, 'name': 'TimedC', 'color': 'blue'},
    #    {'level': 16, 'name': 'TA', 'color': 'blue'},
    #    {'level': 20, 'name': 'GenB', 'color': 'blue'},
    #   {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    #],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}

In [31]:
max_gameday = 14

# Level distribution charts: user counts, percentage share, and day-over-day diff (up to level 30)
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_gameday'] <= max_gameday], 
              x='max_gameday', 
              labels={'max_gameday': 'Player Game Day'},
              y='users',
              color='combined_dimension',
              title='Players game day distribution',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_gameday': True, 'users': True},)

#fig = add_event_annotations(fig, events_config, x_col='max_gameday', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)
#fig = add_median_lines(fig, pl_ftue_funnel_agg, x_col='p50_max_gameday', ftue_col='FTUE_flag', platform_col='platform')

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_gameday'] <= max_gameday], 
              x='max_gameday', 
              labels={'max_gameday': 'Player Game Day'},
              y='pctg_users',
              color='combined_dimension',
              title='Players at each game day (percentage)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_gameday': True, 'users': True},)

#fig = add_event_annotations(fig, events_config, x_col='max_gameday', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=True)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_gameday'] <= max_gameday], 
              x='max_gameday', 
              labels={'max_gameday': 'Player Game Day'},
              y='pctg_diff_users',
              color='combined_dimension',
              title='Players at each game day (percentage diff change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'max_gameday': True, 'users': True},)

#fig = add_event_annotations(fig, events_config, x_col='max_gameday', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg, show_annotations=False)

fig.show()

### Percentile max level reached

In [32]:
def plot_percentile_comparison(df, percentile='all', measure_name='Max Game Day'):
    """
    Plot percentile comparison between FTUE groups.
    
    Args:
        df: DataFrame with columns: days_since_install, FTUE_flag, platform, p10, p50, p90
        percentile: 'all', 'p10', 'p50', or 'p90'
        measure_name: Name of the measure for axis labels (default: 'Max Game Day')
    """
    # Rename columns to generic percentile names if they have suffixes
    df_plot = df.copy()
    for col in df_plot.columns:
        if col.startswith('p') and col[1:].replace('0', '').isdigit():
            # Already in generic format (p10, p50, p90)
            pass
        elif '_' in col and any(col.startswith(f'p{x}_') for x in ['10', '50', '90']):
            # Has suffix like p10_max_gameday -> rename to p10
            percentile_num = col.split('_')[0]
            df_plot = df_plot.rename(columns={col: percentile_num})
    
    if percentile == 'all':
        percentile_cols = ['p10', 'p50', 'p90']
    else:
        percentile_cols = [percentile]
    
    df_melted = df_plot.melt(
        id_vars=['days_since_install', 'FTUE_flag', 'platform'],
        value_vars=percentile_cols,
        var_name='percentile',
        value_name='value'
    )
    
    fig = px.line(
        df_melted,
        x='days_since_install',
        y='value',
        color='FTUE_flag',
        facet_col='platform',
        line_dash='percentile',
        title=f'Percentile comparison - {percentile if percentile != "all" else "P10/P50/P90"}',
        width=1500,
        height=600,
        labels={'value': measure_name, 'days_since_install': 'Days Since Install'}
    )
    
    fig.show()

In [33]:
# Band chart: P10/P50/P90 level progression comparison between A.old and B.new
plot_percentile_comparison(level_pcts_by_group, percentile='all', measure_name='Max Game Day')

# To inspect a single percentile with diff bar:
# plot_percentile_comparison(level_pcts_by_group, percentile='p50')
# plot_percentile_comparison(level_pcts_by_group, percentile='p10')
# plot_percentile_comparison(level_pcts_by_group, percentile='p90')

### Average max level reached

In [34]:
# hide-output
# Compute weighted average max level by FTUE group, platform, and days since install
days_since_install_baseline = 28

data_filtered = processed_data[processed_data['days_since_install'] <= days_since_install_baseline]

pl_ftue_max_level_agg = compute_weighted_progression(data_filtered, measure_col='max_gameday', dimension_cols=['dummy', 'days_since_install','FTUE_flag','platform'], min_bucket_size=10)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff_max_gameday'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['weighted_avg_max_gameday'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'platform', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag','platform'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby(['days_since_install', 'platform'])['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg

,dummy,days_since_install,FTUE_flag,platform,cohort_users,weighted_avg_max_gameday,combined_dimension,pctg_diff_max_gameday,total_cohort_users,pctg_users,pctg_diff_users
0,dummy,0,A.old,AND,6615,1.971523,dummy | A.old,0.000000,6615,1.000000,0.000000
1,dummy,0,A.old,IOS,16466,1.829166,dummy | A.old,0.000000,16466,1.000000,0.000000
2,dummy,0,B.new,AND,4839,2.162179,dummy | B.new,0.096705,4839,1.000000,0.000000
3,dummy,0,B.new,IOS,13983,1.982353,dummy | B.new,0.083747,13983,1.000000,0.000000
4,dummy,1,A.old,AND,2796,3.926554,dummy | A.old,0.000000,6615,0.422676,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
111,dummy,27,B.new,IOS,619,15.955840,dummy | B.new,-0.057006,13983,0.044268,-0.547818
112,dummy,28,A.old,AND,695,21.534527,dummy | A.old,0.000000,6615,0.105064,0.000000
113,dummy,28,A.old,IOS,1577,17.070909,dummy | A.old,0.000000,16466,0.095773,0.000000
114,dummy,28,B.new,AND,133,26.205993,dummy | B.new,0.216929,4839,0.027485,-0.738398


In [35]:
# hide-output
# Bar chart: surviving cohort size at each day since install by FTUE group
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              facet_row='platform',
              width=1200,
              height=800,
              barmode='group',
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},)

fig.show()

In [36]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='combined_dimension',
              title='Player level reached at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_max_gameday',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},)

fig.show()

In [37]:
# hide-output
# Sort data by user and day for per-user progression analysis
data.sort_values(['user_id', 'days_since_install'], inplace=True)
data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version
200921,100186873D03CE09,2026-06-28,2026-06-28,US,2026-06-28,2026-06-01,0,5,3,IOS,Non-Attributed,Non-Attributed,0.78.0
1709,1002854972AF27D8,2026-08-02,2026-08-02,FR,2026-08-02,2026-08-01,0,4,2,AND,Non-Attributed,Non-Attributed,0.80.0
6731,1002854972AF27D8,2026-08-03,2026-08-02,FR,2026-08-02,2026-08-01,1,6,3,AND,Non-Attributed,Non-Attributed,0.80.0
7217,1002854972AF27D8,2026-08-06,2026-08-02,FR,2026-08-02,2026-08-01,4,6,3,AND,Non-Attributed,Non-Attributed,0.80.0
4104,1002854972AF27D8,2026-08-08,2026-08-02,FR,2026-08-02,2026-08-01,6,6,4,AND,Non-Attributed,Non-Attributed,0.80.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
78508,FFFA00CAB8D18E98,2026-06-11,2026-06-02,GB,2026-05-31,2026-06-01,9,15,11,IOS,FACEBOOK,UA,0.76.0
81616,FFFA00CAB8D18E98,2026-06-12,2026-06-02,GB,2026-05-31,2026-06-01,10,15,11,IOS,FACEBOOK,UA,0.76.0
253377,FFFC66E056EBEBDF,2026-07-13,2026-07-13,DE,2026-07-12,2026-07-01,0,1,1,AND,Non-Attributed,Non-Attributed,0.79.0
253238,FFFC66E056EBEBDF,2026-07-14,2026-07-13,DE,2026-07-12,2026-07-01,1,1,1,AND,Non-Attributed,Non-Attributed,0.79.0


## Engagement metrics

In [38]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,4E457DC98BCE17B0,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
1,71519947AC3D8D6F,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
2,EF4D5A89B33B8146,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,4,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
3,B83F360FF8B1D488,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
4,8EE0EDAA76098BD1,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447440,93A74E043E9178B8,2026-05-30,2026-05-30,0,9,6,1,1,<NA>,7,...,0,0,0,0,0,6,0,1,0,2026-06-18 18:29:41.321007+00:00
447441,2BF08A70CB095BDA,2026-05-22,2026-05-30,8,18,13,1,9,1,9,...,0,0,0,0,0,2,0,1,0,2026-06-18 18:29:41.321007+00:00
447442,E347D9EFB0AC9BEC,2026-05-28,2026-05-30,2,14,10,1,3,1,26,...,0,0,0,0,0,2,0,1,0,2026-06-18 18:29:41.321007+00:00
447443,66A8C4D0719CE7E4,2026-05-29,2026-05-30,1,21,17,1,2,1,19,...,0,0,0,0,0,0,0,1,1,2026-06-18 18:29:41.321007+00:00


In [39]:
processed_data

,user_id,dt,install_dt,country_code,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,platform,display_campaign_network,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,4D0D930275F288D5,2026-08-07,2026-08-05,CA,2026-08-02,2026-08-01,2,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
1,891692CD0FACEB18,2026-08-09,2026-08-09,US,2026-08-09,2026-08-01,0,3,2,IOS,Non-Attributed,Non-Attributed,0.81.0,N,B.new,dummy
2,2D0AB22D5B135786,2026-08-06,2026-08-05,DE,2026-08-02,2026-08-01,1,5,3,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
3,F75214BDE9929B0A,2026-08-08,2026-08-02,FR,2026-08-02,2026-08-01,6,6,4,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
4,285D0B14A1C4AEB,2026-08-05,2026-08-05,BR,2026-08-02,2026-08-01,0,2,1,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
296549,ED4C793E1996E209,2026-08-04,2026-07-29,DE,2026-07-26,2026-07-01,6,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296550,5D45FDE6D033ED4E,2026-07-30,2026-07-29,US,2026-07-26,2026-07-01,1,9,6,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296551,6049F381B9B68CBE,2026-08-02,2026-07-29,US,2026-07-26,2026-07-01,4,8,5,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy
296552,5F4E2C7AE2C61852,2026-08-08,2026-07-29,DE,2026-07-26,2026-07-01,10,11,8,IOS,Non-Attributed,Non-Attributed,0.80.0,N,B.new,dummy


In [40]:
days_since_install_limit = 28

engagement_data = abmetrics[['user_id','dt','days_since_install','n_sessions','n_mins_in_game','n_merges','n_tasks_completed']]
engagement_data = processed_data[['user_id','dt','FTUE_flag','platform']].merge(engagement_data, on=['user_id', 'dt'], how='left')
engagement_data = engagement_data[engagement_data['days_since_install'] <= days_since_install_limit]
engagement_data

,user_id,dt,FTUE_flag,platform,days_since_install,n_sessions,n_mins_in_game,n_merges,n_tasks_completed
0,4D0D930275F288D5,2026-08-07,B.new,IOS,2,1,6,97.0,0
1,891692CD0FACEB18,2026-08-09,B.new,IOS,0,1,17,312.0,0
2,2D0AB22D5B135786,2026-08-06,B.new,IOS,1,9,52,516.0,0
3,F75214BDE9929B0A,2026-08-08,B.new,IOS,6,2,2,4.0,0
4,285D0B14A1C4AEB,2026-08-05,B.new,IOS,0,1,4,64.0,0
...,...,...,...,...,...,...,...,...,...
225665,ED4C793E1996E209,2026-08-04,B.new,IOS,6,7,35,654.0,0
225666,5D45FDE6D033ED4E,2026-07-30,B.new,IOS,1,6,94,877.0,0
225667,6049F381B9B68CBE,2026-08-02,B.new,IOS,4,3,39,507.0,0
225668,5F4E2C7AE2C61852,2026-08-08,B.new,IOS,10,13,44,612.0,0


In [41]:
# calculate percentiles P10, P50 and P90 for n_sessions, n_mins_in_game, and n_merges by FTUE_flag, platform, and days_since_install
percentiles = [0.1, 0.5, 0.9]

# Calculate P10, P50, P90 percentiles for n_sessions from engagement_data
percentiles_sessions = engagement_data.groupby(['FTUE_flag', 'platform', 'days_since_install'])['n_sessions'].quantile(percentiles).unstack(fill_value=0).reset_index()
percentiles_sessions.columns = ['FTUE_flag', 'platform', 'days_since_install', 'p10_sessions', 'p50_sessions', 'p90_sessions']
percentiles_sessions


engagement_data_agg = engagement_data.groupby(['FTUE_flag','platform','days_since_install']).agg(
    avg_sessions=('n_sessions', 'mean'),
    avg_n_mins_in_game=('n_mins_in_game', 'mean'),
    avg_n_merges=('n_merges', 'mean'),
    avg_n_tasks_completed=('n_tasks_completed', 'mean'),
    total_users=('user_id', 'nunique')
).reset_index()

engagement_data_agg = engagement_data_agg.merge(percentiles_sessions, on=['FTUE_flag', 'platform', 'days_since_install'], how='left') 

engagement_data_agg.sort_values(['days_since_install','platform','FTUE_flag'], inplace=True)
engagement_data_agg['pctg_diff_avg_sessions'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_sessions'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_mins_in_game'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_mins_in_game'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_merges'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_merges'].pct_change().fillna(0)
engagement_data_agg['pctg_diff_avg_n_tasks_completed'] = engagement_data_agg.groupby(['days_since_install','platform'])['avg_n_tasks_completed'].pct_change().fillna(0)

engagement_data_agg

,FTUE_flag,platform,days_since_install,avg_sessions,avg_n_mins_in_game,avg_n_merges,avg_n_tasks_completed,total_users,p10_sessions,p50_sessions,p90_sessions,pctg_diff_avg_sessions,pctg_diff_avg_n_mins_in_game,pctg_diff_avg_n_merges,pctg_diff_avg_n_tasks_completed
0,A.old,AND,0,2.418111,26.052584,336.487419,7.738737,6637,1.0,2.0,5.0,0.0,0.0,0.000000,0.0
58,B.new,AND,0,2.659404,29.015622,376.722713,0.042549,4865,1.0,2.0,5.0,0.099786,0.113733,0.119574,-0.994502
29,A.old,IOS,0,2.503216,25.424505,366.910790,8.203544,16478,1.0,2.0,5.0,0.0,0.0,0.000000,0.0
87,B.new,IOS,0,2.496464,27.268057,402.579553,0.003787,13997,1.0,2.0,5.0,-0.002698,0.072511,0.097214,-0.999538
1,A.old,AND,1,3.800574,32.772238,396.563845,6.495696,2788,1.0,2.0,9.0,0.0,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,B.new,IOS,27,4.519943,31.769231,357.867521,1.217949,702,1.0,3.0,10.0,-0.194757,-0.269305,-0.246768,-0.696206
28,A.old,AND,28,4.89726,41.839041,495.571918,3.962329,292,1.0,3.0,10.0,0.0,0.0,0.000000,0.0
86,B.new,AND,28,4.47191,39.213483,429.501873,1.441948,267,1.0,3.0,10.0,-0.086855,-0.062754,-0.133321,-0.636086
57,A.old,IOS,28,5.753968,43.771825,476.164683,4.156746,504,1.0,4.0,13.0,0.0,0.0,0.000000,0.0


### Sessions

In [42]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_sessions',
              color='FTUE_flag',
              title='Average sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_sessions',
              color='FTUE_flag',
              title='Average sessions at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()


In [43]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p10_sessions',
              color='FTUE_flag',
              title='10th percentile sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p50_sessions',
              color='FTUE_flag',
              title='Median sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='p90_sessions',
              color='FTUE_flag',
              title='90th percentile sessions at day x since install',
              facet_col='platform',
              width=1500,
              height=400,
              hover_data={'p10_sessions': True, 'p50_sessions': True, 'p90_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()


### Minutes

In [54]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_mins_in_game',
              color='FTUE_flag',
              title='Average minutes in game at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_mins_in_game',
              color='FTUE_flag',
              title='Average minutes in game at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

### Merges

In [55]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_merges',
              color='FTUE_flag',
              title='Average merges at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_merges',
              color='FTUE_flag',
              title='Average merges at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True},)
fig.show()

### Tasks

In [46]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='avg_n_tasks_completed',
              color='FTUE_flag',
              title='Average tasks completed at day x since install',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True, 'avg_n_tasks_completed': True},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(engagement_data_agg, 
              x='days_since_install', 
              y='pctg_diff_avg_n_tasks_completed',
              color='FTUE_flag',
              title='Average tasks completed at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=600,
              hover_data={'avg_sessions': True, 'avg_n_mins_in_game': True, 'avg_n_merges': True, 'avg_n_tasks_completed': True},)
fig.show()

## Economy metrics

In [47]:
abmetrics

,user_id,install_dt,dt,days_since_install,max_level,max_gameday,active,active_cumu,days_since_last_active_exc_today,n_sessions,...,n_cashdash_ms_completed,n_datedash_ms_completed,n_space_ms_completed,n_art_ms_completed,n_roadtrip_ms_completed,n_fortunes_ms_completed,n_timedalbum_sets_completed,reached_600_clash_points,reached_rank1,loading_timestamp
0,4E457DC98BCE17B0,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
1,71519947AC3D8D6F,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
2,EF4D5A89B33B8146,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,4,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
3,B83F360FF8B1D488,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
4,8EE0EDAA76098BD1,2026-07-22,2026-07-22,0,1,1,1,1,<NA>,1,...,0,0,0,0,0,0,0,0,0,2026-07-28 12:45:43.207970+00:00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
447440,93A74E043E9178B8,2026-05-30,2026-05-30,0,9,6,1,1,<NA>,7,...,0,0,0,0,0,6,0,1,0,2026-06-18 18:29:41.321007+00:00
447441,2BF08A70CB095BDA,2026-05-22,2026-05-30,8,18,13,1,9,1,9,...,0,0,0,0,0,2,0,1,0,2026-06-18 18:29:41.321007+00:00
447442,E347D9EFB0AC9BEC,2026-05-28,2026-05-30,2,14,10,1,3,1,26,...,0,0,0,0,0,2,0,1,0,2026-06-18 18:29:41.321007+00:00
447443,66A8C4D0719CE7E4,2026-05-29,2026-05-30,1,21,17,1,2,1,19,...,0,0,0,0,0,0,0,1,1,2026-06-18 18:29:41.321007+00:00


In [48]:
days_since_install_limit = 28

economy_data = abmetrics[['user_id','dt','days_since_install','coins_spent','coins_earned','gems_spent','gems_earned_game','energy_spent','energy_earned_game']]
economy_data = processed_data[['user_id','dt','FTUE_flag','platform']].merge(economy_data, on=['user_id', 'dt'], how='left')
economy_data = economy_data[economy_data['days_since_install'] <= days_since_install_limit]

# safe divide to avoid division by zero
economy_data['coins_sink'] = economy_data['coins_spent']/economy_data['coins_earned'].replace(0, np.nan)
economy_data['gems_sink'] = economy_data['gems_spent']/economy_data['gems_earned_game'].replace(0, np.nan)
economy_data['energy_sink'] = economy_data['energy_spent']/economy_data['energy_earned_game'].replace(0, np.nan)

economy_data

,user_id,dt,FTUE_flag,platform,days_since_install,coins_spent,coins_earned,gems_spent,gems_earned_game,energy_spent,energy_earned_game,coins_sink,gems_sink,energy_sink
0,4D0D930275F288D5,2026-08-07,B.new,IOS,2,40,52,0,0,106,10,0.769231,<NA>,10.6
1,891692CD0FACEB18,2026-08-09,B.new,IOS,0,144,144,0,57,305,301,1.0,0.0,1.013289
2,2D0AB22D5B135786,2026-08-06,B.new,IOS,1,300,313,0,0,506,45,0.958466,<NA>,11.244444
3,F75214BDE9929B0A,2026-08-08,B.new,IOS,6,0,21,0,11,0,0,0.0,0.0,<NA>
4,285D0B14A1C4AEB,2026-08-05,B.new,IOS,0,13,17,0,50,56,100,0.764706,0.0,0.56
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225665,ED4C793E1996E209,2026-08-04,B.new,IOS,6,375,431,30,6,631,161,0.87007,5.0,3.919255
225666,5D45FDE6D033ED4E,2026-07-30,B.new,IOS,1,730,854,10,33,881,520,0.854801,0.30303,1.694231
225667,6049F381B9B68CBE,2026-08-02,B.new,IOS,4,415,399,6,31,508,215,1.0401,0.193548,2.362791
225668,5F4E2C7AE2C61852,2026-08-08,B.new,IOS,10,580,530,10,31,606,55,1.09434,0.322581,11.018182


In [49]:
# calculate percentiles P10, P50 and P90 for n_sessions, n_mins_in_game, and n_merges by FTUE_flag, platform, and days_since_install
percentiles = [0.1, 0.5, 0.9]

# Calculate P10, P50, P90 percentiles for n_sessions from engagement_data
percentiles_sessions = economy_data.groupby(['FTUE_flag', 'platform', 'days_since_install'])['gems_earned_game'].quantile(percentiles).unstack(fill_value=0).reset_index()
percentiles_sessions.columns = ['FTUE_flag', 'platform', 'days_since_install', 'p10_gems_earned_game', 'p50_gems_earned_game', 'p90_gems_earned_game']
percentiles_sessions


economy_data_agg = economy_data.groupby(['FTUE_flag','platform','days_since_install']).agg(
    avg_coins_earned=('coins_earned', 'mean'),
    avg_coins_spent=('coins_spent', 'mean'),
    avg_coins_sink=('coins_sink', 'mean'),
    avg_gems_earned_game=('gems_earned_game', 'mean'),
    avg_gems_spent=('gems_spent', 'mean'),
    avg_gems_sink=('gems_sink', 'mean'),
    avg_energy_earned_game=('energy_earned_game', 'mean'),
    avg_energy_spent=('energy_spent', 'mean'),
    avg_energy_sink=('energy_sink', 'mean'),
    total_users=('user_id', 'nunique')
).reset_index()

economy_data_agg = economy_data_agg.merge(percentiles_sessions, on=['FTUE_flag', 'platform', 'days_since_install'], how='left') 

economy_data_agg.sort_values(['days_since_install','platform','FTUE_flag'], inplace=True)
economy_data_agg['pctg_diff_avg_coins_earned'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_coins_earned'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_coins_spent'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_coins_spent'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_coins_sink'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_coins_sink'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_gems_earned_game'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_gems_earned_game'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_gems_spent'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_gems_spent'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_gems_sink'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_gems_sink'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_energy_earned_game'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_energy_earned_game'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_energy_spent'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_energy_spent'].pct_change().fillna(0)
economy_data_agg['pctg_diff_avg_energy_sink'] = economy_data_agg.groupby(['days_since_install','platform'])['avg_energy_sink'].pct_change().fillna(0)

economy_data_agg

,FTUE_flag,platform,days_since_install,avg_coins_earned,avg_coins_spent,avg_coins_sink,avg_gems_earned_game,avg_gems_spent,avg_gems_sink,avg_energy_earned_game,...,p90_gems_earned_game,pctg_diff_avg_coins_earned,pctg_diff_avg_coins_spent,pctg_diff_avg_coins_sink,pctg_diff_avg_gems_earned_game,pctg_diff_avg_gems_spent,pctg_diff_avg_gems_sink,pctg_diff_avg_energy_earned_game,pctg_diff_avg_energy_spent,pctg_diff_avg_energy_sink
0,A.old,AND,0,191.961428,176.854452,0.770772,55.205665,19.517854,0.322173,253.230526,...,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
58,B.new,AND,0,215.713464,194.926413,0.811992,55.693936,20.31963,0.318207,273.270504,...,62.0,0.123733,0.102186,0.053479,0.008845,0.041079,-0.012311,0.079137,0.120845,0.097299
29,A.old,IOS,0,205.887972,183.221447,0.773774,55.435854,19.974572,0.313198,267.405207,...,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
87,B.new,IOS,0,229.927627,204.70458,0.816688,55.943416,23.499964,0.354323,284.960349,...,66.0,0.116761,0.117252,0.05546,0.009156,0.176494,0.131307,0.06565,0.099846,0.082818
1,A.old,AND,1,279.737805,292.697274,0.957254,9.658178,21.930057,2.992134,137.069225,...,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114,B.new,IOS,27,582.937322,560.440171,1.046928,9.477208,29.509972,3.590277,190.420228,...,31.0,-0.196527,-0.200022,-0.126378,-0.344585,-0.035576,0.502747,-0.269782,-0.184442,0.28752
28,A.old,AND,28,793.589041,696.681507,0.944593,17.003425,30.267123,1.623832,261.136986,...,47.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
86,B.new,AND,28,785.247191,749.205993,1.064586,11.685393,16.749064,1.705015,236.318352,...,32.0,-0.010512,0.075392,0.127031,-0.312762,-0.446625,0.049995,-0.095041,-0.02636,-0.028382
57,A.old,IOS,28,739.678571,720.103175,1.040312,18.39881,24.890873,3.253182,271.392857,...,40.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Coins

In [50]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(economy_data_agg, 
              x='days_since_install', 
              y='avg_coins_sink',
              color='FTUE_flag',
              title='Average coins sink at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_coins_earned': True, 'avg_coins_spent': True, 'avg_coins_sink': True,},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(economy_data_agg,
              x='days_since_install', 
              y='pctg_diff_avg_coins_sink',
              color='FTUE_flag',
              title='Average coins sink at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_coins_earned': True, 'avg_coins_spent': True, 'avg_coins_sink': True},)
fig.show()

### Gems

In [51]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(economy_data_agg, 
              x='days_since_install', 
              y='avg_gems_sink',
              color='FTUE_flag',
              title='Average gems sink at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_gems_earned_game': True, 'avg_gems_spent': True, 'avg_gems_sink': True,},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(economy_data_agg,
              x='days_since_install', 
              y='pctg_diff_avg_gems_sink',
              color='FTUE_flag',
              title='Average gems sink at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_gems_earned_game': True, 'avg_gems_spent': True, 'avg_gems_sink': True},)
fig.show()

### Energy

In [52]:
# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(economy_data_agg, 
              x='days_since_install', 
              y='avg_energy_sink',
              color='FTUE_flag',
              title='Average energy sink at day x since install',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_energy_earned_game': True, 'avg_energy_spent': True, 'avg_energy_sink': True,},)
fig.show()

# Weighted average max level and daily pct-change progression by FTUE group and platform
fig = px.line(economy_data_agg,
              x='days_since_install', 
              y='pctg_diff_avg_energy_sink',
              color='FTUE_flag',
              title='Average energy sink at day x since install (pct change)',
              facet_col='platform',
              width=1500,
              height=350,
              hover_data={'avg_energy_earned_game': True, 'avg_energy_spent': True, 'avg_energy_sink': True},)
fig.show()

In [53]:
# hide-output
# Export notebook to HTML for sharing and archival
export_notebook_html(
    notebook_path='./storyday_1-20.ipynb',
    output_path='./storyday_1-20.html',
)

Saved to storyday_1-20.html


PosixPath('storyday_1-20.html')